In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/AI Engineering")

# Verify the directory
print("Project exists:", PROJECT_DIR.exists())
print("Evaluation folder exists:", (PROJECT_DIR / "evaluation").exists())

# Set working directory
os.chdir(PROJECT_DIR)

# Add project root to Python's module search path
sys.path.insert(0, str(PROJECT_DIR))

print("Current directory:", os.getcwd())
print("Project files:", os.listdir(PROJECT_DIR))

Project exists: True
Evaluation folder exists: True
Current directory: /content/drive/MyDrive/AI Engineering
Project files: ['knowledge-base', 'implementation', 'vector_db', '.gradio', 'evaluation']


In [3]:
!pip install -q litellm

In [4]:
!pip install -q langchain_chroma

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.2 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.2 which is incompatible.
tweepy 4.17.0 requires oauthlib<4,>=3.2.0, but you have oauthlib 4.0.1 which is incompatible.


In [5]:
!pip install -q litellm pydantic python-dotenv langchain-chroma langchain-core langchain-openai langchain-huggingface chromadb openai gradio pandas sentence-transformers

In [6]:
!pip show langchain-core langchain-openai langchain-chroma langchain-huggingface

Name: langchain-core
Version: 1.8.4
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: certifi, httpx, jsonpatch, langchain-protocol, langsmith, packaging, pydantic, pyyaml, tenacity, typing-extensions, uuid-utils
Required-by: langchain, langchain-chroma, langchain-huggingface, langchain-openai, langgraph, langgraph-checkpoint, langgraph-prebuilt, langgraph-sdk
---
Name: langchain-openai
Version: 1.8.5
Summary: An integration package connecting OpenAI and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/openai
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.13/dist-packages
Requires: certifi, langchain-core, openai, tiktoken
Required-by: 
---
Name: langchain-chroma
Version: 1.1.0
Summary: An integration package connecting Chroma and LangChain.
Home-page: https://docs.langchain.com/oss/p

In [7]:
from google.colab import userdata
import os

os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

print("API key loaded:", bool(os.environ["OPENROUTER_API_KEY"]))

API key loaded: True


In [8]:
from evaluation.eval import evaluate_all_retrieval, evaluate_all_answers

print("Both evaluation functions imported successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (4 outputs left out)

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Both evaluation functions imported successfully!


In [9]:
import gradio as gr
import pandas as pd
from collections import defaultdict
from dotenv import load_dotenv

from evaluation.eval import evaluate_all_retrieval, evaluate_all_answers

load_dotenv(override=True)

# Color coding thresholds - Retrieval
MRR_GREEN = 0.9
MRR_AMBER = 0.75
NDCG_GREEN = 0.9
NDCG_AMBER = 0.75
COVERAGE_GREEN = 90.0
COVERAGE_AMBER = 75.0

# Color coding thresholds - Answer (1-5 scale)
ANSWER_GREEN = 4.5
ANSWER_AMBER = 4.0


def get_color(value: float, metric_type: str) -> str:
    """Get color based on metric value and type."""
    if metric_type == "mrr":
        if value >= MRR_GREEN:
            return "green"
        elif value >= MRR_AMBER:
            return "orange"
        else:
            return "red"
    elif metric_type == "ndcg":
        if value >= NDCG_GREEN:
            return "green"
        elif value >= NDCG_AMBER:
            return "orange"
        else:
            return "red"
    elif metric_type == "coverage":
        if value >= COVERAGE_GREEN:
            return "green"
        elif value >= COVERAGE_AMBER:
            return "orange"
        else:
            return "red"
    elif metric_type in ["accuracy", "completeness", "relevance"]:
        if value >= ANSWER_GREEN:
            return "green"
        elif value >= ANSWER_AMBER:
            return "orange"
        else:
            return "red"
    return "black"


def format_metric_html(
    label: str,
    value: float,
    metric_type: str,
    is_percentage: bool = False,
    score_format: bool = False,
) -> str:
    """Format a metric with color coding."""
    color = get_color(value, metric_type)
    if is_percentage:
        value_str = f"{value:.1f}%"
    elif score_format:
        value_str = f"{value:.2f}/5"
    else:
        value_str = f"{value:.4f}"
    return f"""
    <div style="margin: 10px 0; padding: 15px; background-color: #f5f5f5; border-radius: 8px; border-left: 5px solid {color};">
        <div style="font-size: 14px; color: #666; margin-bottom: 5px;">{label}</div>
        <div style="font-size: 28px; font-weight: bold; color: {color};">{value_str}</div>
    </div>
    """


def run_retrieval_evaluation(progress=gr.Progress()):
    """Run retrieval evaluation and yield updates."""
    total_mrr = 0.0
    total_ndcg = 0.0
    total_coverage = 0.0
    category_mrr = defaultdict(list)
    count = 0

    for test, result, prog_value in evaluate_all_retrieval():
        count += 1
        total_mrr += result.mrr
        total_ndcg += result.ndcg
        total_coverage += result.keyword_coverage

        category_mrr[test.category].append(result.mrr)

        # Update progress bar only
        progress(prog_value, desc=f"Evaluating test {count}...")

    # Calculate final averages
    avg_mrr = total_mrr / count
    avg_ndcg = total_ndcg / count
    avg_coverage = total_coverage / count

    # Create final summary metrics HTML
    final_html = f"""
    <div style="padding: 0;">
        {format_metric_html("Mean Reciprocal Rank (MRR)", avg_mrr, "mrr")}
        {format_metric_html("Normalized DCG (nDCG)", avg_ndcg, "ndcg")}
        {format_metric_html("Keyword Coverage", avg_coverage, "coverage", is_percentage=True)}
        <div style="margin-top: 20px; padding: 10px; background-color: #d4edda; border-radius: 5px; text-align: center; border: 1px solid #c3e6cb;">
            <span style="font-size: 14px; color: #155724; font-weight: bold;">✓ Evaluation Complete: {count} tests</span>
        </div>
    </div>
    """

    # Create final bar chart data
    category_data = []
    for category, mrr_scores in category_mrr.items():
        avg_cat_mrr = sum(mrr_scores) / len(mrr_scores)
        category_data.append({"Category": category, "Average MRR": avg_cat_mrr})

    df = pd.DataFrame(category_data)

    return final_html, df


def run_answer_evaluation(progress=gr.Progress()):
    """Run answer evaluation and yield updates (async)."""
    total_accuracy = 0.0
    total_completeness = 0.0
    total_relevance = 0.0
    category_accuracy = defaultdict(list)
    count = 0

    for test, result, prog_value in evaluate_all_answers():
        count += 1
        total_accuracy += result.accuracy
        total_completeness += result.completeness
        total_relevance += result.relevance

        category_accuracy[test.category].append(result.accuracy)

        # Update progress bar only
        progress(prog_value, desc=f"Evaluating test {count}...")

    # Calculate final averages
    avg_accuracy = total_accuracy / count
    avg_completeness = total_completeness / count
    avg_relevance = total_relevance / count

    # Create final summary metrics HTML
    final_html = f"""
    <div style="padding: 0;">
        {format_metric_html("Accuracy", avg_accuracy, "accuracy", score_format=True)}
        {format_metric_html("Completeness", avg_completeness, "completeness", score_format=True)}
        {format_metric_html("Relevance", avg_relevance, "relevance", score_format=True)}
        <div style="margin-top: 20px; padding: 10px; background-color: #d4edda; border-radius: 5px; text-align: center; border: 1px solid #c3e6cb;">
            <span style="font-size: 14px; color: #155724; font-weight: bold;">✓ Evaluation Complete: {count} tests</span>
        </div>
    </div>
    """

    # Create final bar chart data
    category_data = []
    for category, accuracy_scores in category_accuracy.items():
        avg_cat_accuracy = sum(accuracy_scores) / len(accuracy_scores)
        category_data.append({"Category": category, "Average Accuracy": avg_cat_accuracy})

    df = pd.DataFrame(category_data)

    return final_html, df


def main():
    """Launch the Gradio evaluation app."""
    theme = gr.themes.Soft(font=["Inter", "system-ui", "sans-serif"])

    with gr.Blocks(title="RAG Evaluation Dashboard", theme=theme) as app:
        gr.Markdown("# 📊 RAG Evaluation Dashboard")
        gr.Markdown("Evaluate retrieval and answer quality for the Insurellm RAG system")

        # RETRIEVAL SECTION
        gr.Markdown("## 🔍 Retrieval Evaluation")

        retrieval_button = gr.Button("Run Evaluation", variant="primary", size="lg")

        with gr.Row():
            with gr.Column(scale=1):
                retrieval_metrics = gr.HTML(
                    "<div style='padding: 20px; text-align: center; color: #999;'>Click 'Run Evaluation' to start</div>"
                )

            with gr.Column(scale=1):
                retrieval_chart = gr.BarPlot(
                    x="Category",
                    y="Average MRR",
                    title="Average MRR by Category",
                    y_lim=[0, 1],
                    height=400,
                )

        # ANSWERING SECTION
        gr.Markdown("## 💬 Answer Evaluation")

        answer_button = gr.Button("Run Evaluation", variant="primary", size="lg")

        with gr.Row():
            with gr.Column(scale=1):
                answer_metrics = gr.HTML(
                    "<div style='padding: 20px; text-align: center; color: #999;'>Click 'Run Evaluation' to start</div>"
                )

            with gr.Column(scale=1):
                answer_chart = gr.BarPlot(
                    x="Category",
                    y="Average Accuracy",
                    title="Average Accuracy by Category",
                    y_lim=[1, 5],
                    height=400,
                )

        # Wire up the evaluations
        retrieval_button.click(
            fn=run_retrieval_evaluation,
            outputs=[retrieval_metrics, retrieval_chart],
        )

        answer_button.click(
            fn=run_answer_evaluation,
            outputs=[answer_metrics, answer_chart],
        )

    app.launch(inbrowser=True)


if __name__ == "__main__":
    main()

/tmp/ipykernel_1271/4213963700.py:177: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title="RAG Evaluation Dashboard", theme=theme) as app:


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6474e7d5860f74e96a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
from litellm import completion
from pydantic import BaseModel, Field
import os

class TestResponse(BaseModel):
    feedback: str
    accuracy: float

response = completion(
    model="openrouter/baidu/ernie-5.5-flash-preview:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    messages=[
        {"role": "user", "content": "Evaluate this answer: Paris is the capital of France."}
    ],
    response_format=TestResponse
)

print(response.choices[0].message.content)

{"accuracy": 1, "feedback": "This answer is correct. Paris is the capital of France. It has served as the country's capital and cultural, political, and economic center for centuries. The statement is factually accurate with no errors or ambiguities."}


In [ ]:
import importlib
import evaluation.eval as eval_module

# Reload the edited Python file
importlib.reload(eval_module)

# Set the correct model
eval_module.MODEL = "openrouter/baidu/ernie-5.5-flash-preview:free"

# Inspect the reloaded function
import inspect
print(inspect.getsource(eval_module.evaluate_answer))

def evaluate_answer(test: TestQuestion) -> tuple[AnswerEval, str, list]:
    """
    Evaluate answer quality using LLM-as-a-judge (async).

    Args:
        test: TestQuestion object containing question and reference answer

    Returns:
        Tuple of (AnswerEval object, generated_answer string, retrieved_docs list)
    """
    # Get RAG response using shared answer module
    generated_answer, retrieved_docs = answer_question(test.question)

    # LLM judge prompt
    judge_messages = [
        {
            "role": "system",
            "content": "You are an expert evaluator assessing the quality of answers. Evaluate the generated answer by comparing it to the reference answer. Only give 5/5 scores for perfect answers.",
        },
        {
            "role": "user",
            "content": f"""Question:
{test.question}

Generated Answer:
{generated_answer}

Reference Answer:
{test.reference_answer}

Please evaluate the generated answer on three dimensions:
1. Accuracy: How fac

In [ ]:
from evaluation.eval import evaluate_answer
from evaluation.test import load_tests

tests = load_tests()

result = evaluate_answer(tests[0])

print("Evaluation:", result[0])
print("Generated answer:", result[1])
print("Retrieved documents:", len(result[2]))

Evaluation: feedback="The answer identifies the correct winner (Maxine) and the correct awarding organization (Insurellm) and the correct year (2023), so the core fact is right. However, the award name is rendered as the 'Insurellm IIOTY Innovator Award', which garbles the reference name 'Insurellm Innovator of the Year (IIOTY) award' — it treats the acronym as an adjective and never expands it, which a reader unfamiliar with the acronym would find confusing. The quoted supporting sentence is reproduced faithfully, which suggests the answer is simply inheriting the source document's own imprecise phrasing rather than misreading it; that mitigates but does not eliminate the naming error.\n\nOn completeness, the answer covers who, what, and when, but omits the winner's surname ('Maxine Thompson'), which the reference answer includes. It also fails to expand IIOTY, leaving the reader with less information than the reference conveys. These are small gaps rather than substantive omissions.\

In [ ]:
from evaluation.eval import evaluate_answer
from evaluation.test import load_tests

tests = load_tests()
result = evaluate_answer(tests[0])

print("Evaluation:", result[0])
print("Generated answer:", result[1])
print("Retrieved documents:", len(result[2]))

In [ ]:
import evaluation.eval as eval_module
from evaluation.test import load_tests

# Override the model in the imported evaluation module
eval_module.MODEL = "openrouter/baidu/ernie-5.5-flash-preview:free"

print("Model being used:", eval_module.MODEL)

tests = load_tests()

result = eval_module.evaluate_answer(tests[0])

print("\nEvaluation:", result[0])
print("Generated answer:", result[1])
print("Retrieved documents:", len(result[2]))

Model being used: openrouter/baidu/ernie-5.5-flash-preview:free

Evaluation: feedback='The answer correctly identifies **Maxine** as the winner and correctly names Insurellm as the awarding organization, so the core fact is right. However, it drops her surname (**Thompson**), which the reference answer includes — this matters because "Maxine" alone is only partially identifying. It also garbles the award name: it renders IIOTY as "Insurellm IIOTY Innovator Award," which both duplicates the acronym and mis-expands it. The reference gives the correct expansion, "Insurellm Innovator of the Year (IIOTY)." Two inaccuracies of this kind in a short answer keep it short of perfect. It is not wholly wrong, so a 1 is not warranted, but it is not fully accurate either.' accuracy=3.0 completeness=2.0 relevance=5.0
Generated answer: According to the context, **Maxine** won the prestigious Insurellm IIOTY Innovator Award in 2023.
Retrieved documents: 10


In [ ]:
from pydantic import BaseModel
from litellm import completion
import os

class TestEval(BaseModel):
    feedback: str
    accuracy: int
    completeness: int
    relevance: int

response = completion(
    model="openrouter/baidu/ernie-5.5-flash-preview:free",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    messages=[
        {
            "role": "user",
            "content": """
            Evaluate this answer:
            Question: What is RAG?
            Answer: RAG combines retrieval with generation.
            Return feedback, accuracy, completeness, and relevance.
            """
        }
    ],
    response_format=TestEval
)

print(response.choices[0].message.content)

{"accuracy": 4, "completeness": 2, "feedback": "The answer correctly identifies the two core components of RAG (Retrieval-Augmented Generation): a retrieval step and a generation step. This is factually correct as far as it goes, but it's a minimal, surface-level definition that stops at the name without explaining how the two parts work together or why the technique exists. To be complete, it should explain that relevant documents are fetched from an external knowledge source (often a vector database) and injected into the prompt as context, allowing a language model to generate grounded answers rather than relying solely on its parametric memory. The main benefit (reducing hallucination and enabling up-to-date or domain-specific knowledge without retraining) is also missing.", "relevance": 5}


In [ ]:
import evaluation.eval as eval_module
import importlib

importlib.reload(eval_module)

from evaluation.test import load_tests

tests = load_tests()

print(result)

(AnswerEval(feedback='The answer correctly identifies **Maxine** as the winner and correctly names Insurellm as the awarding organization, so the core fact is right. However, it drops her surname (**Thompson**), which the reference answer includes — this matters because "Maxine" alone is only partially identifying. It also garbles the award name: it renders IIOTY as "Insurellm IIOTY Innovator Award," which both duplicates the acronym and mis-expands it. The reference gives the correct expansion, "Insurellm Innovator of the Year (IIOTY)." Two inaccuracies of this kind in a short answer keep it short of perfect. It is not wholly wrong, so a 1 is not warranted, but it is not fully accurate either.', accuracy=3.0, completeness=2.0, relevance=5.0), 'According to the context, **Maxine** won the prestigious Insurellm IIOTY Innovator Award in 2023.', [Document(id='09512619-96ae-4ccf-a395-c1aa5f827f94', metadata={'doc_type': 'employees', 'source': '/content/drive/MyDrive/AI Engineering/knowledg

In [11]:
import importlib
import implementation.answer as answer_module
import evaluation.eval as eval_module

# Reload the updated answer module
importlib.reload(answer_module)

# Make evaluation use the refreshed answer module
eval_module.answer_question = answer_module.answer_question

# Test the first evaluation case
from evaluation.test import load_tests

tests = load_tests()
result = eval_module.evaluate_answer(tests[0])

print("Evaluation result:")
print(result[0])

print("\nGenerated answer:")
print(result[1])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

OpenAIRateLimitError: Error code: 429 - {'error': {'message': 'Rate limit exceeded: free-models-per-day. Add 10 credits to unlock 1000 free model requests per day: https://openrouter.ai/settings/credits?utm_source=api&utm_campaign=free_daily_limit', 'code': 429, 'metadata': {'headers': {'X-RateLimit-Limit': '50', 'X-RateLimit-Remaining': '0', 'X-RateLimit-Reset': '1795219200000'}, 'limit_source': 'openrouter_free_tier_daily', 'remedy_hint': 'Wait for the daily reset (see X-RateLimit-Reset), or purchase credits to raise your free-model daily limit.', 'provider_name': None}}, 'user_id': 'user_3KVIsY2jO6wnaoyk0wG1PDpfl30'}